# Phase 7 — Ablation Studies & Final Results

Systematic ablation analysis of the multimodal fusion system:
1. Single-modality vs full fusion
2. GAT vs No-GAT
3. Multi-task vs single-task
4. Per-ticker breakdown
5. Cross-phase consolidated comparison

In [2]:
import json, sys
from pathlib import Path

ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(ROOT))

p4 = json.loads((ROOT / "models/phase4_results.json").read_text())
p5 = json.loads((ROOT / "models/phase5_results.json").read_text())
p6 = json.loads((ROOT / "models/phase6_results.json").read_text())
p7 = json.loads((ROOT / "models/phase7_ablation_results.json").read_text())

checks_passed = 0
checks_total = 0

def CHECK(name, cond):
    global checks_passed, checks_total
    checks_total += 1
    status = "PASS" if cond else "FAIL"
    if cond: checks_passed += 1
    print(f"  [{status}] {name}")
    return cond

print("All result files loaded.")

All result files loaded.


## 1. Single-Modality Ablation

In [ ]:
print("=== Single-Modality Ablation (Direction AUC, test split) ===\n")
sm = p7["single_modality_ablation"]
fusion_auc = p6["fusion"]["direction_metrics"]["auc"]

print(f"  {'Modality':<12s}  {'AUC':>7s}  {'vs Fusion':>10s}")
print("  " + "-" * 35)
for mod in ["price", "gat", "doc",
            "surprise_feat"]:  # Tests 5-d backward-looking earnings features as gating input
    key = "surprise" if mod == "surprise_feat" else mod
    auc = sm[key]["auc"]
    delta = auc - fusion_auc
    print(f"  {mod:<12s}  {auc:.4f}    {delta:+.4f}")
print(f"  {'FULL FUSION':<12s}  {fusion_auc:.4f}")

CHECK("all single-modality AUC values recorded", len(sm) >= 4)
# GAT alone > price alone — graph adds cross-company information
CHECK("GAT alone outperforms price alone", sm["gat"]["auc"] > sm["price"]["auc"])
print("\n  Key finding: GAT alone (0.528) outperforms price alone (0.510),")
print("  confirming that graph attention captures cross-company signals.")

## 2. GAT vs No-GAT Ablation

In [4]:
print("=== GAT vs No-GAT Ablation ===\n")
ga = p7["gat_ablation"]
print(f"  With GAT:    AUC = {ga['with_gat_auc']:.4f}")
print(f"  Without GAT: AUC = {ga['without_gat_auc']:.4f}")
print(f"  Delta:       {ga['delta']:+.4f}")

CHECK("GAT improves over no-GAT", ga["delta"] > 0)
CHECK("delta is meaningful (> 0.01)", ga["delta"] > 0.01)
print("\n  Graph attention adds +0.023 AUC over identical architecture without graph layers.")
print("  The inter-company signal propagation is a genuine contribution.")

=== GAT vs No-GAT Ablation ===

  With GAT:    AUC = 0.5257
  Without GAT: AUC = 0.5027
  Delta:       +0.0231
  [PASS] GAT improves over no-GAT
  [PASS] delta is meaningful (> 0.01)

  Graph attention adds +0.023 AUC over identical architecture without graph layers.
  The inter-company signal propagation is a genuine contribution.


## 3. Multi-Task vs Single-Task

In [ ]:
print("=== Multi-Task vs Single-Task ===\n")
mt = p7["multitask_comparison"]
print(f"  Single-task direction AUC:  {mt['single_task_direction_auc']:.4f}")
print(f"  Multi-task direction AUC:   {mt['multitask_direction_auc']:.4f}")
print(f"  Multi-task volatility R²:   {mt['multitask_volatility_r2']:.4f}")
# Surprise prediction head removed in V2 due to data leakage

CHECK("multi-task volatility R² > 0.30", mt["multitask_volatility_r2"] > 0.30)
CHECK("single-task vs multi-task direction AUC gap < 0.02",
      abs(mt["single_task_direction_auc"] - mt["multitask_direction_auc"]) < 0.02)

print("\n  Trade-off: Multi-task direction AUC is slightly lower (-0.008),")
print("  but gains strong volatility prediction (R²=0.41).")
print("  The combined model provides direction + volatility forecasts — richer output than single-task.")

## 4. Per-Ticker Analysis

In [6]:
print("=== Per-Ticker Direction AUC (Fusion Model) ===\n")
pt = p7["per_ticker_analysis"]

print(f"  {'Ticker':>6s}  {'AUC':>7s}  {'N':>6s}  {'UP%':>6s}  Signal")
print("  " + "-" * 45)
above_random = 0
for ticker in sorted(pt.keys()):
    t = pt[ticker]
    signal = "✓" if t["auc"] > 0.52 else "~" if t["auc"] > 0.50 else "✗"
    if t["auc"] > 0.50:
        above_random += 1
    print(f"  {ticker:>6s}  {t['auc']:.4f}  {t['n_samples']:6d}  {t['up_pct']:.1f}%  {signal}")

CHECK(f"{above_random}/10 tickers above random", above_random >= 5)

best = max(pt, key=lambda k: pt[k]["auc"])
worst = min(pt, key=lambda k: pt[k]["auc"])
print(f"\n  Best:  {best} (AUC={pt[best]['auc']:.4f})")
print(f"  Worst: {worst} (AUC={pt[worst]['auc']:.4f})")
print(f"  {above_random}/10 tickers have AUC > 0.50")

=== Per-Ticker Direction AUC (Fusion Model) ===

  Ticker      AUC       N     UP%  Signal
  ---------------------------------------------
    AAPL  0.4736    1062  54.0%  ✗
     AMD  0.5113    1062  53.5%  ~
    AMZN  0.5720    1062  55.6%  ✓
   GOOGL  0.4952    1062  58.8%  ✗
    INTC  0.5653    1062  46.3%  ✓
    META  0.5289    1062  57.4%  ✓
    MSFT  0.5734    1062  56.9%  ✓
    NVDA  0.5562    1062  60.3%  ✓
    ORCL  0.5248    1062  51.2%  ✓
    TSLA  0.4739    1062  48.6%  ✗
  [PASS] 7/10 tickers above random

  Best:  MSFT (AUC=0.5734)
  Worst: AAPL (AUC=0.4736)
  7/10 tickers have AUC > 0.50


## 5. Cross-Phase Consolidated Table

In [ ]:
print("=== Consolidated Results ===\n")

print("--- Direction Prediction (AUC) ---")
cp = p7["cross_phase_direction_auc"]
# Surprise (MLP) removed: evidenced data leakage (AUC=0.6313 was spurious).
# News (FinBERT+BiGRU) removed: news modality dropped in V2 (2.1% coverage).
excluded_models = {"Surprise (MLP)", "News (FinBERT+BiGRU)"}
filtered_cp = {k: v for k, v in cp.items() if k not in excluded_models}
best_auc = max(filtered_cp.values())
print(f"  {'Model':<30s}  {'AUC':>7s}")
print("  " + "-" * 40)
for name, auc in sorted(filtered_cp.items(), key=lambda x: -x[1]):
    marker = " ◀ best" if auc == best_auc else ""
    print(f"  {name:<30s}  {auc:.4f}{marker}")

print(f"\n--- Volatility Prediction ---")
vm = p6["fusion"]["volatility_metrics"]
print(f"  R² = {vm['r2']:.4f},  RMSE = {vm['rmse']:.4f},  MAE = {vm['mae']:.4f}")

print(f"\n--- Gate Weights ---")
gw = p7["fusion_gate_weights"]
for name, w in sorted(gw.items(), key=lambda x: -x[1]):
    bar = '█' * int(w * 40)
    print(f"  {name:<10s}  {w:.4f}  {bar}")

CHECK("results JSON saved", (ROOT / "models/phase7_ablation_results.json").exists())

print(f"\n{'='*50}")
print(f"  CHECKS PASSED: {checks_passed}/{checks_total}")
print(f"{'='*50}")